경로 설정 및 모듈 임포트

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# 1. 상위 폴더의 generators.py를 참조하기 위한 경로 설정
sys.path.append(os.path.abspath(os.path.join('..')))

# 2. 설계한 클래스들 임포트
from generators import Environment, BirdDyn, DroneDyn, TrajectoryGenerator

# 3. 주피터 노트북 시각화 설정
%matplotlib inline
%config InlineBackend.figure_format = 'retina' # 고해상도 그래프 설정

print("✅ 시뮬레이션 엔진 및 라이브러리 로드 완료")

1. 환경 설정 (Environment)
    * fps=30 ▶ 30 프레임
    * wind_speed=3.0 ▶ 바람 세기 3m/s

2. Agent 초기화 (Initilazation)
    * start_pos=[0, 80, 100] ▶ 비행체 시작 지점 고도 100미터
    * pigeon, drone ▶ Agent 객체화

3. 시뮬레이션 제어 변수
    * frames=300 ▶ 총 비행 프레임수 (30FPS 기준 10초)
    * p_history_2d, d_history_2d ▶ 관측 데이터 저장소

In [ ]:
# 1. 환경 설정 (측풍 3m/s, 30FPS)
env = Environment(fps=30, wind_speed=3.0, goal_pos=[200.0, 120.0, 80.0])

# 2. 에이전트 초기화 (동일한 지점에서 시작)
# 비둘기(Bird)와 드론(Drone) 생성
start_pos = [0, 80, 100] # x, y, z
pigeon = BirdDyn(env, species="pigeon", start_pos=start_pos)
drone = DroneDyn(env, model="quadcopter", start_pos=start_pos)

# 3. 데이터 저장용 리스트
frames = 300
p_history_2d = []
d_history_2d = []

# 4. 시뮬레이션 루프 실행
for i in range(frames):
    # (1) 물리 엔진 구동 (3D 좌표 업데이트)
    pigeon.step()
    drone.step()
    
    # (2) 가상 카메라 투영 (2D JSON 데이터 추출)
    p_history_2d.append(pigeon.get_observation(i))
    d_history_2d.append(drone.get_observation(i))

print(f"✅ {frames} 프레임 시뮬레이션 완료 (약 {frames/30:.1f}초 비행)")

# 궤적 생성 및 시각화 (3D 시각화)

1. 3D 도화지 준비 (Canvas Setup)
    * projection='3d' ▶ mplot3d 툴킷을 이용해 3차원 공간 생성
    * figsize=(10, 8) ▶ 그래프 크기 설정

2. 데이터 가공 및 선 그리기
    * np.array(pigeon.history) 
        ▶ 리스트 형태의 히스토리 → Numpy 배열
        ▶ 모든 프레임의 좌표 슬라이싱

3. 주요 지점 마킹 (Markers)
    * ax.scatter 
        ▶ Start : 출발점(0,0,100) 검은색 점 찍기
        ▶ Goal : 도착점(default 300,300,50) 녹색 점 찍기

4. 관측 시점 
    * view_init(elev=25, azim=-45) ▶ 카메라의 앙각과 방위각

5. 라벨링
    * ax.set_... ▶ 각 축이 실제 물리량인 미터(m) 단위임을 명시

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# 비둘기 3D 경로 (파란색)
p_3d = np.array(pigeon.history)
ax.plot(p_3d[:, 0], p_3d[:, 1], p_3d[:, 2], 'b-', label='Pigeon (Bird)', alpha=0.8)

# 드론 3D 경로 (빨간색)
d_3d = np.array(drone.history)
ax.plot(d_3d[:, 0], d_3d[:, 1], d_3d[:, 2], 'r-', label='Drone', alpha=0.8)

# 목표 지점 및 시작점 표시
ax.scatter(0, 80, 100, color='black', s=50, label='Start')
ax.scatter(env.x_goal[0], env.x_goal[1], env.x_goal[2], color='green', marker='*', s=200, label='Goal')

ax.set_title("3D Trajectory Comparison (Physical Space)")
ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)"); ax.set_zlabel("Altitude (m)")
ax.legend()
ax.view_init(elev=25, azim=-45) # 보기 좋은 각도로 조정
plt.show()

# 2D 투영 및 관측 데이터 시각화 (XZ Mapping)

1. 2D 도화지 구성 (Subplot Setup)
    * plt.figure(figsize=(16, 7)) ▶ 가로로 긴 형태의 전체 그래프 크기 설정
    * plt.subplot(1, 2, n) ▶ 1행 2열 구조로 비둘기(좌)와 드론(우)의 관측 화면을 독립적으로 배치

2. 정규화 좌표 추출 (Coordinate Extraction)
    * [pt['cx'] for pt in history] ▶ 정규화된 가로 중심 좌표(cx) 리스트 생성
    * [pt['cy'] for pt in history] ▶ 정규화된 세로 중심 좌표(cy) 리스트 생성 (고도 반영)

3. 궤적 및 박스 시각화 (Rendering)
    * plt.plot(cxs, cys, ...) ▶ 0.0 ~ 1.0 범위 내에서 비행체가 그려낸 2D 이동 경로 시각화
    * for i in range(0, len(history), 40) ▶ 데이터가 너무 겹치지 않도록 40프레임 간격으로 바운딩 박스 샘플링

4. 중심점 보정 및 배치 (Centering & Patching)
    * patches.Rectangle ▶ 계산된 w, h를 이용해 객체의 탐지 영역(BBox) 생성
    * (cx - w/2, cy - h/2) ▶ 사각형의 기준점(좌하단)을 중심 좌표(cx, cy) 기준으로 오프셋 보정
    * ax.add_patch(rect) ▶ 생성된 박스 객체를 도화지에 추가

5. 축 매핑 및 반전 (Axis Mapping)
    * plt.xlim(0, 1); plt.ylim(1, 0) ▶ 화면 규격(0 ~ 1) 고정 및 고도가 높을수록 위쪽(0)에 위치하도록 Y축 반전
    * plt.grid(True, linestyle=':') ▶ 정규화된 좌표 위치를 쉽게 파악할 수 있도록 격자 표시

In [ ]:
import matplotlib.patches as patches  

def draw_xz_projection(history, title, color):
    ax = plt.gca()
    cxs = [pt['cx'] for pt in history]
    cys = [pt['cy'] for pt in history] # 고도 반영
    
    # 궤적 그리기
    plt.plot(cxs, cys, color=color, linewidth=2, label='XZ Path')

    # 바운딩 박스 (실제 거리감 확인)
    for i in range(0, len(history), 40):
        pt = history[i]
        cx, cy, w, h = pt['cx'], pt['cy'], pt['w'], pt['h']
        # Rectangle은 좌하단 기준이므로 중심점 보정
        rect = patches.Rectangle((cx - w/2, cy - h/2), w, h, 
                                 edgecolor=color, facecolor=color, alpha=0.3)
        ax.add_patch(rect)

    plt.title(title)
    plt.xlabel("X (Forward)"); plt.ylabel("Z (Altitude)")
    plt.xlim(0, 1); plt.ylim(1, 0) # 1이 지면, 0이 하늘
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend()

plt.figure(figsize=(16, 7))
plt.subplot(1, 2, 1); draw_xz_projection(p_history_2d, "Pigeon Side-View (XZ Mapping)", "blue")
plt.subplot(1, 2, 2); draw_xz_projection(d_history_2d, "Drone Side-View (XZ Mapping)", "red")
plt.show()